In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'PRODUCT_CATEGORY_TRANSLATION_ENRICHED'

In [0]:

PCT_df = spark.table('SALES.BRONZE.PRODUCT_CATEGORY_TRANSLATION')
PCT_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

PCT_enriched_df = PCT_df.select(
    upper(trim(col('product_category_name'))).alias('product_category_name'),
    upper(trim(col('product_category_name_english'))).alias('product_category_name_english'),
    col('file_name'),
    col('file_path'),
    col('ingestion_timestamp').alias('load_timestamp')
).filter('product_category_name IS NOT NULL')

display(PCT_enriched_df.limit(1)) 

In [0]:
PCT_enriched_df.createOrReplaceTempView('enriched_productcategorytranslation_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY product_category_name
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_productcategorytranslation_temp_vw
  )
  WHERE rn = 1
) src
ON (src.product_category_name  = tgt.product_category_name)
WHEN MATCHED THEN
  UPDATE SET
    tgt.product_category_name = src.product_category_name,
    tgt.product_category_name_english = src.product_category_name_english
    
WHEN NOT MATCHED THEN
  INSERT (
    product_category_name,
    product_category_name_english,
    file_name,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.product_category_name,
    src.product_category_name_english,
    src.file_name,
    src.file_path,
    current_timestamp()
  )
""")